In [2]:
import torch
print(torch.cuda.is_available()) # Must return True


ModuleNotFoundError: No module named 'torch'

In [18]:
import fitz
from PIL import Image
import io

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption

import torch
import torch.nn.functional as F
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image

In [7]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.mode = "accurate"

In [8]:
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

In [9]:
result = converter.convert("pdfs/government-data-security-policies.pdf")
doc = result.document

[INFO] 2026-04-13 16:21:07,333 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-13 16:21:07,334 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-13 16:21:07,344 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-13 16:21:07,345 [RapidOCR] main.py:50: Using C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-13 16:21:07,465 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-13 16:21:07,466 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-13 16:21:07,467 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-13 16:21:07,467 [RapidOCR] main.py:50: Using C:\Users\UserAdmin\Documents\RAG\venv\Lib\site-packages\rapidocr\models\ch_pto

In [12]:
md_index = []

In [13]:
for page_number, page in doc.pages.items():
    page_md = doc.export_to_markdown(page_no=page_number)
    md_index.append({
        "page_number": page_number,
        "page_md": page_md
    }) 

In [16]:
fitz_doc = fitz.open("pdfs/government-data-security-policies.pdf")

In [ ]:
model_id = "Qwen/Qwen2-VL-7B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16, 
    device_map="cuda"
)
processor = AutoProcessor.from_pretrained(model_id)

In [17]:
vector_index = []

In [ ]:
for i in range(len(fitz_doc)):
    page = fitz_doc.load_page(i)
    pix = page.get_pixmap(dpi=300)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    
